# Lab 6.12 &mdash; Capstone: A Guardrailed LangChain Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Vet tools against an allow-list before binding them
- Assemble a real create_agent with a system prompt + recursion cap
- Run the guardrailed agent over a suite of real tasks and read the traces

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; Choosing a framework](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-06-12")
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Capstone &mdash; a real **synthesis** of the module. You compose the pieces into one guarded agent: an
**allow-list** so only vetted tools are bound, a **system prompt** that sets the agent's policy, a
**`recursion_limit`** cap so the loop can never run away, and a **tracing callback** (Lab 11) so every
step is recorded. Then you run it **for real** over a small task suite &mdash; the bridge to Day 4.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key, e.g. 'capital of france'."""
    return {"population of metropolis": "120000", "capital of france": "Paris"}.get(key.lower().strip(), "unknown")

print("a vetted tool:", lookup.name)

## Build it
Complete `vet_tools` (the allow-list guardrail), `build_agent` (the real `create_agent` with a system
prompt), and `run_config` (the recursion cap).

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langchain.agents import create_agent

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key, e.g. 'capital of france'."""
    facts = {"population of metropolis": "120000", "capital of france": "Paris"}
    return facts.get(key.lower().strip(), "unknown")

@tool
def calculator(expression: str) -> str:
    """Do exact arithmetic on an expression such as 120000/2."""
    try:
        return str(safe_calc(expression))
    except Exception:
        return "error: not a numeric expression"

ALLOWED = {"lookup", "calculator"}
SYSTEM = "You are a careful assistant. Use the tools for facts and math; never guess a number."

class TraceHandler(BaseCallbackHandler):
    def __init__(self): self.events = []
    def on_tool_end(self, output, **kw): self.events.append(str(output)[:60])

def vet_tools(tools):
    return [t for t in tools if ___]   # TODO: keep only allow-listed tools (guardrail)

def build_agent():
    tools = vet_tools([lookup, calculator])
    return create_agent(llm, tools, system_prompt=___)   # TODO: pass the system prompt

def run_config(max_steps=8, handler=None):
    cfg = ___   # TODO: {"recursion_limit": max_steps} -- cap the loop
    if handler is not None:
        cfg["callbacks"] = [handler]
    return cfg

try:
    agent = build_agent()
    print("agent type :", type(agent).__name__)
    print("bound tools:", [t.name for t in vet_tools([lookup, calculator])])
    print("dropped a fake risky tool?:", "delete_db" not in [t.name for t in vet_tools([lookup, calculator])])
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the guardrailed agent over a real task suite. Each item is a fresh real agent run; the callback records every tool observation.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        SUITE = [
            "What is the capital of France?",
            "What is the population of metropolis, divided by 2?",
            "What is 15 times 3?",
        ]
        for q in SUITE:
            handler = TraceHandler()
            result = agent.invoke({"messages": [("user", q)]}, config=run_config(8, handler))
            print("Q:", q)
            print("  final :", result["messages"][-1].content)
            print("  tools :", handler.events)
            print()
        print("That was a REAL guardrailed LangChain agent over a suite. Next: Day 4 -- task automation & multi-agent teams.")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The agent is a real **`CompiledStateGraph`** built from **vetted** tools only &mdash; a fake `delete_db` tool never gets bound.
- The **system prompt** shapes behaviour ("never guess a number"); the **recursion cap** bounds the loop; the **callback** records every tool result.
- Across the suite you can see which tasks needed tools and which the model answered directly &mdash; all from real traces.

## Your turn (open task &mdash; no grader)
Add your own tool (respecting the allow-list &mdash; add its name to `ALLOWED`), extend `SUITE` with a task
that needs it, and re-run. **What good looks like:** your new tool is vetted in, the agent calls it on the
right task, the recursion cap holds, and the callback logs every step. That's a shippable, guardrailed
agent &mdash; the foundation for Day 4.

---
### Key takeaway
You assembled a guardrailed LangChain agent -- vetted tools, a system prompt, create_agent and a recursion cap -- and ran it over a real suite. That is a shippable agent; next, Day 4 puts agents to work.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>